# Comparaison Fine-tuning CamemBERT Base vs CamemBERT NER

Ce notebook compare les performances de deux modèles de base :
- **CamemBERT Base** (`camembert-base`)
- **CamemBERT NER** (`Jean-Baptiste/camembert-ner`)

Sur deux tâches :
1. **Classification d'intent** (TRIP, NOT_TRIP, UNKNOWN)
2. **Extraction d'entités NER** (departure, destination, intermediate)

**Total : 4 fine-tunings**

| Modèle | Classification | NER |
|--------|---------------|-----|
| camembert-base | ✓ | ✓ |
| camembert-ner | ✓ | ✓ |

**Dataset:** `datasets/augmented/` (STT 100k avec erreurs speech-to-text simulées)

## 1. Installation et imports

In [1]:
!pip install transformers datasets accelerate evaluate seqeval scikit-learn pandas torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import json
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import evaluate
import warnings
warnings.filterwarnings('ignore')

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch version: 2.9.0+cu126
GPU: Tesla T4


In [3]:
# Modèles à comparer
MODELS = {
    "camembert-base": "camembert-base",
    "camembert-ner": "Jean-Baptiste/camembert-ner"
}

# Stockage des résultats
results_classification = {}
results_ner = {}

## 2. Chargement et préparation des données

In [5]:
# Chargement des datasets pré-splittés depuis datasets/augmented/
train_df = pd.read_csv("./datasets/augmented/train.csv")
val_df = pd.read_csv("./datasets/augmented/val.csv")
test_df = pd.read_csv("./datasets/augmented/test.csv")

print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

print(f"\nColonnes: {train_df.columns.tolist()}")
print(f"\nDistribution des intents (train):")
print(train_df['intent'].value_counts())
print(f"\nDistribution des langues (train):")
print(train_df['language'].value_counts())
train_df.head(10)

Train: 69998 samples
Val: 14997 samples
Test: 15005 samples

Colonnes: ['sentence_id', 'sentence', 'intent', 'language', 'departure', 'destination', 'intermediate']

Distribution des intents (train):
intent
TRIP        48998
NOT_TRIP    17500
UNKNOWN      3500
Name: count, dtype: int64

Distribution des langues (train):
language
FRENCH     53200
ENGLISH     7000
UNKNOWN     4200
GERMAN      2274
SPANISH     2274
ITALIAN     1050
Name: count, dtype: int64


,sentence_id,sentence,intent,language,departure,destination,intermediate
0,STT042863,Partir de Pleudihen-sur-Rence vers Longpont,TRIP,FRENCH,Pleudihen-sur-Rance,Longpont,NaN
1,STT074646,Bonsoir,NOT_TRIP,FRENCH,NaN,NaN,NaN
2,STT092070,Sorry,NOT_TRIP,ENGLISH,NaN,NaN,NaN
3,STT017968,Il y a des correspondances de Royat - Chamaliè...,TRIP,FRENCH,Royat - Chamalières,Bolquère-Eyne,NaN
4,STT004528,Alors euh Chaingy Fourneaux Plage a Ormoy-Villers,TRIP,FRENCH,Chaingy Fourneaux Plage,Ormoy-Villers,NaN
5,STT043265,Ch'veux aller de Crépieux la Pape a Lison...,TRIP,FRENCH,Crépieux la Pape,Lison,NaN
6,STT036789,Euh un train de L'Herbergement - Les Brouzils ...,TRIP,FRENCH,L'Herbergement - Les Brouzils,Toulouse Matabiau,NaN
7,STT011131,Bordel je dois tu vois aller de ben Illfurth a...,TRIP,FRENCH,Illfurth,Digoin,NaN
8,STT020788,Dans la mesure du possible hmm Sainte-Pazanne ...,TRIP,FRENCH,Sainte-Pazanne,Brioude,NaN
9,STT053678,trajet combs-la-ville - quincy retournac bièvres,TRIP,FRENCH,Combs-la-Ville - Quincy,Bièvres,Retournac


In [6]:
# Nettoyage des données
def clean_dataframe(df):
    df = df.copy()
    df = df.dropna(subset=['sentence', 'intent'])
    df['sentence'] = df['sentence'].astype(str).str.strip()
    df = df[df['sentence'].str.len() > 0]

    # Remplacement des NaN pour departure/destination/intermediate
    df['departure'] = df['departure'].fillna('')
    df['destination'] = df['destination'].fillna('')
    df['intermediate'] = df['intermediate'].fillna('')

    return df

train_df = clean_dataframe(train_df)
val_df = clean_dataframe(val_df)
test_df = clean_dataframe(test_df)

print(f"Train après nettoyage: {len(train_df)}")
print(f"Val après nettoyage: {len(val_df)}")
print(f"Test après nettoyage: {len(test_df)}")
print(f"\nDistribution des intents (train):")
print(train_df['intent'].value_counts())

Train après nettoyage: 69866
Val après nettoyage: 14970
Test après nettoyage: 14980

Distribution des intents (train):
intent
TRIP        48998
NOT_TRIP    17500
UNKNOWN      3368
Name: count, dtype: int64


---
# PARTIE A : Classification d'Intent (TRIP, NOT_TRIP, UNKNOWN)

Fine-tuning des deux modèles sur la tâche de classification à 3 classes

In [7]:
# Configuration pour la classification (3 classes)
intent_labels = ["NOT_TRIP", "TRIP", "UNKNOWN"]
label2id_cls = {label: i for i, label in enumerate(intent_labels)}
id2label_cls = {i: label for i, label in enumerate(intent_labels)}

print(f"Labels de classification: {intent_labels}")
print(f"label2id: {label2id_cls}")

Labels de classification: ['NOT_TRIP', 'TRIP', 'UNKNOWN']
label2id: {'NOT_TRIP': 0, 'TRIP': 1, 'UNKNOWN': 2}


In [8]:
# Métriques pour classification
def compute_metrics_classification(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [15]:
def train_classification_model(model_name, model_path, train_df, val_df, test_df):
    """
    Fine-tune un modèle pour la classification d'intent
    """
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION - {model_name}")
    print(f"{'='*60}")

    output_dir = f"../models/{model_name}-classification"

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Préparation des datasets
    def prepare_dataset(df):
        return Dataset.from_dict({
            'text': df['sentence'].tolist(),
            'label': [label2id_cls[l] for l in df['intent'].tolist()]
        })

    dataset = DatasetDict({
        'train': prepare_dataset(train_df),
        'validation': prepare_dataset(val_df),
        'test': prepare_dataset(test_df)
    })

    # Tokenization
    def tokenize(examples):
        return tokenizer(examples['text'], padding=False, truncation=True, max_length=128)

    tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])

    # Modèle (3 classes)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(intent_labels),
        id2label=id2label_cls,
        label2id=label2id_cls,
        ignore_mismatched_sizes=True
    )
    model.to(DEVICE)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=256,
        per_device_eval_batch_size=256,
        num_train_epochs=5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{output_dir}/logs",
        logging_steps=100,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics_classification,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # Entraînement
    trainer.train()

    # Évaluation
    results = trainer.evaluate(tokenized['test'])
    print(f"\nRésultats sur test set:")
    for key, value in results.items():
        print(f"  {key}: {value:.4f}")

    # Sauvegarde
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    return results, model, tokenizer

In [16]:
# Fine-tuning Classification - CamemBERT Base
results_cls_base, model_cls_base, tokenizer_cls_base = train_classification_model(
    "camembert-base",
    MODELS["camembert-base"],
    train_df, val_df, test_df
)
results_classification["camembert-base"] = results_cls_base


CLASSIFICATION - camembert-base


Map:   0%|          | 0/69866 [00:00<?, ? examples/s]

Map:   0%|          | 0/14970 [00:00<?, ? examples/s]

Map:   0%|          | 0/14980 [00:00<?, ? examples/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.185200,0.026251,0.998664,0.998667,0.998677,0.998664
2,0.014100,0.007161,0.999466,0.999466,0.999466,0.999466
3,0.006800,0.003909,0.999666,0.999666,0.999666,0.999666
4,0.004300,0.002715,0.999866,0.999866,0.999866,0.999866
5,0.004400,0.002597,0.999800,0.999800,0.999800,0.999800



Résultats sur test set:
  eval_loss: 0.0035
  eval_accuracy: 0.9995
  eval_f1: 0.9995
  eval_precision: 0.9995
  eval_recall: 0.9995
  eval_runtime: 6.8573
  eval_samples_per_second: 2184.5330
  eval_steps_per_second: 8.6040
  epoch: 5.0000


In [17]:
!ls -la ./results/

ls: cannot access './results/': No such file or directory


In [18]:
# Fine-tuning Classification - CamemBERT NER
results_cls_ner, model_cls_ner, tokenizer_cls_ner = train_classification_model(
    "camembert-ner",
    MODELS["camembert-ner"],
    train_df, val_df, test_df
)
results_classification["camembert-ner"] = results_cls_ner


CLASSIFICATION - camembert-ner


tokenizer_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/892 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

Map:   0%|          | 0/14980 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Jean-Baptiste/camembert-ner and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.117500,0.017231,0.998597,0.998598,0.998601,0.998597
2,0.010900,0.005729,0.999466,0.999466,0.999466,0.999466
3,0.007700,0.004563,0.999532,0.999532,0.999532,0.999532
4,0.007100,0.004527,0.999332,0.999332,0.999332,0.999332
5,0.004500,0.003788,0.999399,0.999399,0.999399,0.999399



Résultats sur test set:
  eval_loss: 0.0039
  eval_accuracy: 0.9994
  eval_f1: 0.9994
  eval_precision: 0.9994
  eval_recall: 0.9994
  eval_runtime: 6.6312
  eval_samples_per_second: 2259.0320
  eval_steps_per_second: 8.8970
  epoch: 5.0000


---
# PARTIE B : Extraction d'Entités NER (departure, destination, intermediate)

Fine-tuning des deux modèles sur la tâche NER avec 3 types d'entités

In [19]:
# Configuration NER (avec intermediate)
ner_labels = ["O", "B-DEP", "I-DEP", "B-DEST", "I-DEST", "B-INTER", "I-INTER"]
label2id_ner = {label: i for i, label in enumerate(ner_labels)}
id2label_ner = {i: label for i, label in enumerate(ner_labels)}

print(f"Labels NER: {ner_labels}")

Labels NER: ['O', 'B-DEP', 'I-DEP', 'B-DEST', 'I-DEST', 'B-INTER', 'I-INTER']


In [20]:
# Fonction pour créer les annotations NER
def create_ner_labels(text, departure, destination, intermediate, tokenizer):
    """
    Crée les labels NER pour chaque token.
    Gère departure, destination et intermediate.
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=128,
        padding=False
    )

    tokens = encoding['input_ids']
    offsets = encoding['offset_mapping']
    labels = [label2id_ner["O"]] * len(tokens)

    text_lower = text.lower()

    def find_and_label_entity(entity, b_label, i_label):
        if not entity or pd.isna(entity) or entity == '':
            return

        entity_lower = entity.lower()
        start_idx = text_lower.find(entity_lower)

        if start_idx == -1:
            # Essayer correspondance partielle
            for i in range(len(entity_lower), 2, -1):
                partial = entity_lower[:i]
                start_idx = text_lower.find(partial)
                if start_idx != -1:
                    entity_lower = partial
                    break

        if start_idx == -1:
            return

        end_idx = start_idx + len(entity_lower)

        is_first = True
        for idx, (token_start, token_end) in enumerate(offsets):
            if token_start is None or token_end is None:
                continue
            if token_start == token_end:
                continue

            if token_start < end_idx and token_end > start_idx:
                if is_first:
                    labels[idx] = label2id_ner[b_label]
                    is_first = False
                else:
                    labels[idx] = label2id_ner[i_label]

    # Labelliser les 3 types d'entités
    find_and_label_entity(departure, "B-DEP", "I-DEP")
    find_and_label_entity(destination, "B-DEST", "I-DEST")
    find_and_label_entity(intermediate, "B-INTER", "I-INTER")

    return {
        'input_ids': tokens,
        'attention_mask': encoding['attention_mask'],
        'labels': labels
    }

In [21]:
# Filtrer les données pour NER (uniquement TRIP avec entités)
train_ner_df = train_df[train_df['intent'] == 'TRIP'].copy()
val_ner_df = val_df[val_df['intent'] == 'TRIP'].copy()
test_ner_df = test_df[test_df['intent'] == 'TRIP'].copy()

print(f"Train NER: {len(train_ner_df)} samples")
print(f"Val NER: {len(val_ner_df)} samples")
print(f"Test NER: {len(test_ner_df)} samples")

# Stats sur les intermediates
n_with_inter = (train_ner_df['intermediate'] != '').sum()
print(f"\nTrain avec intermediate: {n_with_inter} ({100*n_with_inter/len(train_ner_df):.1f}%)")

Train NER: 48998 samples
Val NER: 10498 samples
Test NER: 10504 samples

Train avec intermediate: 7274 (14.8%)


In [22]:
# Métriques NER
seqeval = evaluate.load("seqeval")

def compute_metrics_ner(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        true_label = []
        true_pred = []
        for p, l in zip(prediction, label):
            if l != -100:
                true_label.append(id2label_ner[l])
                true_pred.append(id2label_ner[p])
        true_labels.append(true_label)
        true_predictions.append(true_pred)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [23]:
def train_ner_model(model_name, model_path, train_df, val_df, test_df):
    """
    Fine-tune un modèle pour l'extraction d'entités NER
    """
    print(f"\n{'='*60}")
    print(f"NER - {model_name}")
    print(f"{'='*60}")

    output_dir = f"../models/{model_name}-ner"

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Préparation des datasets
    def prepare_dataset(df):
        data = {'input_ids': [], 'attention_mask': [], 'labels': []}

        for _, row in df.iterrows():
            result = create_ner_labels(
                row['sentence'],
                row['departure'],
                row['destination'],
                row['intermediate'],
                tokenizer
            )
            data['input_ids'].append(result['input_ids'])
            data['attention_mask'].append(result['attention_mask'])
            data['labels'].append(result['labels'])

        return Dataset.from_dict(data)

    print("Préparation des datasets NER...")
    dataset = DatasetDict({
        'train': prepare_dataset(train_df),
        'validation': prepare_dataset(val_df),
        'test': prepare_dataset(test_df)
    })

    # Modèle
    model = AutoModelForTokenClassification.from_pretrained(
        model_path,
        num_labels=len(ner_labels),
        id2label=id2label_ner,
        label2id=label2id_ner,
        ignore_mismatched_sizes=True
    )
    model.to(DEVICE)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=256,
        per_device_eval_batch_size=256,
        num_train_epochs=5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_dir=f"{output_dir}/logs",
        logging_steps=100,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['validation'],
        tokenizer=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=compute_metrics_ner,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # Entraînement
    trainer.train()

    # Évaluation
    results = trainer.evaluate(dataset['test'])
    print(f"\nRésultats sur test set:")
    for key, value in results.items():
        print(f"  {key}: {value:.4f}")

    # Sauvegarde
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    return results, model, tokenizer

In [24]:
# Fine-tuning NER - CamemBERT Base
results_ner_base, model_ner_base, tokenizer_ner_base = train_ner_model(
    "camembert-base",
    MODELS["camembert-base"],
    train_ner_df, val_ner_df, test_ner_df
)
results_ner["camembert-base"] = results_ner_base


NER - camembert-base
Préparation des datasets NER...


Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.427400,0.390412,0.947955,0.964555,0.956183,0.981690
2,0.360800,0.239628,0.957092,0.970267,0.963635,0.984649
3,0.231700,0.176455,0.964146,0.972047,0.968080,0.987538
4,0.175700,0.149775,0.968564,0.975231,0.971886,0.989149
5,0.153500,0.142156,0.968256,0.975465,0.971847,0.989374



Résultats sur test set:
  eval_loss: 0.1529
  eval_precision: 0.9657
  eval_recall: 0.9747
  eval_f1: 0.9702
  eval_accuracy: 0.9883
  eval_runtime: 7.7972
  eval_samples_per_second: 1347.1540
  eval_steps_per_second: 5.3870
  epoch: 5.0000


In [25]:
# Fine-tuning NER - CamemBERT NER
results_ner_ner, model_ner_ner, tokenizer_ner_ner = train_ner_model(
    "camembert-ner",
    MODELS["camembert-ner"],
    train_ner_df, val_ner_df, test_ner_df
)
results_ner["camembert-ner"] = results_ner_ner


NER - camembert-ner
Préparation des datasets NER...


Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at Jean-Baptiste/camembert-ner and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([5]) in the checkpoint and torch.Size([7]) in the model instantiated
- classifier.weight: found shape torch.Size([5, 768]) in the checkpoint and torch.Size([7, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.284300,0.214779,0.930792,0.954675,0.942582,0.979743
2,0.203900,0.126845,0.959526,0.970174,0.964820,0.987506
3,0.128400,0.096362,0.964863,0.973311,0.969068,0.989529
4,0.098500,0.084409,0.968881,0.976729,0.972789,0.990192
5,0.086700,0.080265,0.969972,0.977057,0.973501,0.990551



Résultats sur test set:
  eval_loss: 0.0839
  eval_precision: 0.9672
  eval_recall: 0.9757
  eval_f1: 0.9715
  eval_accuracy: 0.9900
  eval_runtime: 7.7638
  eval_samples_per_second: 1352.9420
  eval_steps_per_second: 5.4100
  epoch: 5.0000


---
# PARTIE C : Comparaison des Résultats

In [26]:
# Tableau comparatif - Classification
print("\n" + "="*70)
print("COMPARAISON - CLASSIFICATION D'INTENT (TRIP, NOT_TRIP, UNKNOWN)")
print("="*70)

cls_comparison = pd.DataFrame({
    'Modèle': ['camembert-base', 'camembert-ner'],
    'Accuracy': [
        results_classification['camembert-base']['eval_accuracy'],
        results_classification['camembert-ner']['eval_accuracy']
    ],
    'F1': [
        results_classification['camembert-base']['eval_f1'],
        results_classification['camembert-ner']['eval_f1']
    ],
    'Precision': [
        results_classification['camembert-base']['eval_precision'],
        results_classification['camembert-ner']['eval_precision']
    ],
    'Recall': [
        results_classification['camembert-base']['eval_recall'],
        results_classification['camembert-ner']['eval_recall']
    ]
})

print(cls_comparison.to_string(index=False))

# Meilleur modèle
best_cls = cls_comparison.loc[cls_comparison['F1'].idxmax(), 'Modèle']
print(f"\n→ Meilleur modèle (Classification): {best_cls}")


COMPARAISON - CLASSIFICATION D'INTENT (TRIP, NOT_TRIP, UNKNOWN)
        Modèle  Accuracy       F1  Precision   Recall
camembert-base  0.999533 0.999533   0.999533 0.999533
 camembert-ner  0.999399 0.999399   0.999400 0.999399

→ Meilleur modèle (Classification): camembert-base


In [27]:
# Tableau comparatif - NER
print("\n" + "="*70)
print("COMPARAISON - EXTRACTION D'ENTITÉS NER (departure, destination, intermediate)")
print("="*70)

ner_comparison = pd.DataFrame({
    'Modèle': ['camembert-base', 'camembert-ner'],
    'Accuracy': [
        results_ner['camembert-base']['eval_accuracy'],
        results_ner['camembert-ner']['eval_accuracy']
    ],
    'F1': [
        results_ner['camembert-base']['eval_f1'],
        results_ner['camembert-ner']['eval_f1']
    ],
    'Precision': [
        results_ner['camembert-base']['eval_precision'],
        results_ner['camembert-ner']['eval_precision']
    ],
    'Recall': [
        results_ner['camembert-base']['eval_recall'],
        results_ner['camembert-ner']['eval_recall']
    ]
})

print(ner_comparison.to_string(index=False))

# Meilleur modèle
best_ner = ner_comparison.loc[ner_comparison['F1'].idxmax(), 'Modèle']
print(f"\n→ Meilleur modèle (NER): {best_ner}")


COMPARAISON - EXTRACTION D'ENTITÉS NER (departure, destination, intermediate)
        Modèle  Accuracy       F1  Precision   Recall
camembert-base  0.988298 0.970192   0.965689 0.974737
 camembert-ner  0.989972 0.971459   0.967240 0.975715

→ Meilleur modèle (NER): camembert-ner


In [28]:
# Résumé final
print("\n" + "="*70)
print("RÉSUMÉ FINAL")
print("="*70)

summary = pd.DataFrame({
    'Tâche': ['Classification', 'Classification', 'NER', 'NER'],
    'Modèle': ['camembert-base', 'camembert-ner', 'camembert-base', 'camembert-ner'],
    'F1 Score': [
        results_classification['camembert-base']['eval_f1'],
        results_classification['camembert-ner']['eval_f1'],
        results_ner['camembert-base']['eval_f1'],
        results_ner['camembert-ner']['eval_f1']
    ]
})

print(summary.to_string(index=False))

print(f"\n" + "-"*70)
print(f"Meilleur modèle pour Classification: {best_cls}")
print(f"Meilleur modèle pour NER: {best_ner}")


RÉSUMÉ FINAL
         Tâche         Modèle  F1 Score
Classification camembert-base  0.999533
Classification  camembert-ner  0.999399
           NER camembert-base  0.970192
           NER  camembert-ner  0.971459

----------------------------------------------------------------------
Meilleur modèle pour Classification: camembert-base
Meilleur modèle pour NER: camembert-ner


In [29]:
# Sauvegarde des résultats
all_results = {
    'classification': {
        'camembert-base': {k: float(v) for k, v in results_classification['camembert-base'].items()},
        'camembert-ner': {k: float(v) for k, v in results_classification['camembert-ner'].items()}
    },
    'ner': {
        'camembert-base': {k: float(v) for k, v in results_ner['camembert-base'].items()},
        'camembert-ner': {k: float(v) for k, v in results_ner['camembert-ner'].items()}
    },
    'best_models': {
        'classification': best_cls,
        'ner': best_ner
    },
    'dataset_info': {
        'source': 'datasets/augmented/',
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df),
        'train_ner_size': len(train_ner_df),
        'intent_labels': intent_labels,
        'ner_labels': ner_labels
    }
}

with open('../models/comparison_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Résultats sauvegardés dans: ../models/comparison_results.json")

Résultats sauvegardés dans: ../models/comparison_results.json


---
# PARTIE D : Tests d'Inférence

In [30]:
# Fonctions d'inférence
def predict_classification(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()

    return {
        'intent': id2label_cls[predicted_class],
        'confidence': predictions[0][predicted_class].item()
    }

def predict_ner(text, model, tokenizer):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop('offset_mapping')[0].tolist()
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().tolist()

    entities = {'departure': [], 'destination': [], 'intermediate': []}
    current_entity = None
    current_text = ""

    for idx, (pred, (start, end)) in enumerate(zip(predictions, offset_mapping)):
        if start == end:
            continue

        label = id2label_ner[pred]
        token_text = text[start:end]

        if label.startswith('B-'):
            if current_entity and current_text:
                entities[current_entity].append(current_text.strip())
            if 'DEP' in label:
                current_entity = 'departure'
            elif 'DEST' in label:
                current_entity = 'destination'
            elif 'INTER' in label:
                current_entity = 'intermediate'
            current_text = token_text
        elif label.startswith('I-') and current_entity:
            current_text += token_text
        else:
            if current_entity and current_text:
                entities[current_entity].append(current_text.strip())
            current_entity = None
            current_text = ""

    if current_entity and current_text:
        entities[current_entity].append(current_text.strip())

    return {
        'departure': ' '.join(entities['departure']) if entities['departure'] else None,
        'destination': ' '.join(entities['destination']) if entities['destination'] else None,
        'intermediate': ' '.join(entities['intermediate']) if entities['intermediate'] else None
    }

In [31]:
# Tests comparatifs avec exemples du dataset augmenté
test_sentences = [
    "Je voudrais un billet de Paris à Lyon",
    "Quelle heure est-il ?",
    "euh ben train marseille toulouse",
    "Je cherche une pizza",
    "Direction Nice au départ de Bordeaux via Montpellier",
    "Bonjour, comment ça va ?",
    "De Paris a Marseille en passant par Lyon",
    "I need a ticket from Nancy to Montbarrey",
    "puis... [coupure]",
    "Ben euh Aubigny-en-Artois vers Harfleur",
    "Bordel je dois tu vois aller de ben Illfurth alors a Digoin",
    "Train de Cantin a Angerville passant par Louhossoa"
]

print("\n" + "="*80)
print("TESTS D'INFÉRENCE COMPARATIFS")
print("="*80)

for sentence in test_sentences:
    print(f"\n→ '{sentence}'")
    print("-" * 70)

    # Classification
    cls_base = predict_classification(sentence, model_cls_base, tokenizer_cls_base)
    cls_ner = predict_classification(sentence, model_cls_ner, tokenizer_cls_ner)

    print(f"  Classification:")
    print(f"    camembert-base: {cls_base['intent']} ({cls_base['confidence']:.2%})")
    print(f"    camembert-ner:  {cls_ner['intent']} ({cls_ner['confidence']:.2%})")

    # NER (seulement si prédit comme TRIP)
    if cls_base['intent'] == 'TRIP' or cls_ner['intent'] == 'TRIP':
        ner_base = predict_ner(sentence, model_ner_base, tokenizer_ner_base)
        ner_ner = predict_ner(sentence, model_ner_ner, tokenizer_ner_ner)

        print(f"  NER:")
        print(f"    camembert-base: DEP={ner_base['departure']}, DEST={ner_base['destination']}, VIA={ner_base['intermediate']}")
        print(f"    camembert-ner:  DEP={ner_ner['departure']}, DEST={ner_ner['destination']}, VIA={ner_ner['intermediate']}")


TESTS D'INFÉRENCE COMPARATIFS

→ 'Je voudrais un billet de Paris à Lyon'
----------------------------------------------------------------------
  Classification:
    camembert-base: TRIP (99.85%)
    camembert-ner:  TRIP (99.86%)
  NER:
    camembert-base: DEP=Paris, DEST=Lyon, VIA=None
    camembert-ner:  DEP=Paris, DEST=Lyon, VIA=None

→ 'Quelle heure est-il ?'
----------------------------------------------------------------------
  Classification:
    camembert-base: NOT_TRIP (99.73%)
    camembert-ner:  NOT_TRIP (99.72%)

→ 'euh ben train marseille toulouse'
----------------------------------------------------------------------
  Classification:
    camembert-base: TRIP (99.76%)
    camembert-ner:  TRIP (99.82%)
  NER:
    camembert-base: DEP=marseille, DEST=toulouse, VIA=None
    camembert-ner:  DEP=marseille, DEST=toulouse, VIA=None

→ 'Je cherche une pizza'
----------------------------------------------------------------------
  Classification:
    camembert-base: NOT_TRIP (99.

---
# Résumé

## Dataset utilisé
- **Source:** `datasets/augmented/` (STT 100k)
- **Train:** ~70k samples
- **Val:** ~15k samples
- **Test:** ~15k samples
- **Caractéristiques:** Erreurs STT simulées (Whisper), multi-langues (FR, EN, ES, DE, IT)

## Modèles entraînés (4 au total)

| Modèle | Tâche | Chemin |
|--------|-------|--------|
| camembert-base | Classification (3 classes) | `models/camembert-base-classification` |
| camembert-base | NER (7 labels) | `models/camembert-base-ner` |
| camembert-ner | Classification (3 classes) | `models/camembert-ner-classification` |
| camembert-ner | NER (7 labels) | `models/camembert-ner-ner` |

## Labels
- **Classification:** TRIP, NOT_TRIP, UNKNOWN
- **NER:** O, B-DEP, I-DEP, B-DEST, I-DEST, B-INTER, I-INTER

## Fichier de résultats
`models/comparison_results.json`